## Bronze to Silver: Data Cleansing and Transformation

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

In [0]:
df = spark.table(f'{catalog_name}.bronze.brz_order_items')
df.show(5)

In [0]:
df.printSchema()

In [0]:
# Transformation: Drop any duplicates
df = df.dropDuplicates(["order_id","item_seq"])

# Convert Two - 2 and cast to Integer
df = df.withColumn(
    'quantity',
    F.when(F.col('quantity')=='Two',2).otherwise(F.col('quantity').cast(IntegerType()))
)

# Remove any $ or any symbol from unit_price and Keep only numeric

df = df.withColumn(
    'unit_price',
    F.regexp_replace('unit_price',"[$]", "").cast('double')
)

# Remove % from discount_pct and cast to double
df = df.withColumn(
    'discount_pct',
    F.regexp_replace('discount_pct',"[%]", "").cast('double')
)

#coupon code processing (convert to lower)
df = df.withColumn(
    'coupon_code',
    F.lower(F.trim(F.col('coupon_code')))
)

#channel Processing
df = df.withColumn(
    'channel',
    F.when(F.col('channel')=='web' , 'Website') \
    .when(F.col("channel")=='app', 'Mobile') \
    .otherwise(F.col('channel'))
)

In [0]:
display(df.limit(5))

In [0]:
# Datatype conversions
# convert dt (String to date)
df = df.withColumn(
    'dt',
    F.to_date(F.col('dt'), 'yyyy-MM-dd')
)

# Convert order_ts (String to timestamp)
df = df.withColumn(
    'order_ts',
    F.coalesce(
        F.to_timestamp(F.col('order_ts'), 'yyyy-MM-dd HH:mm:ss'),
        F.to_timestamp(F.col('order_ts'), 'dd-MM-yyyy HH:mm')
    )
)

# convert item_seq (string - integer)
df = df.withColumn(
    'item_seq',
    F.col("item_seq").cast(IntegerType())
)

#convert tax amount (string - double, strip non-numeric characheters)
df = df.withColumn(
    'tax_amount',
    F.regexp_replace('tax_amount', r'[^0-9.\-]',"").cast('double')
)

# Add processed time
df = df.withColumn(
    'processed_time',
    F.current_timestamp()
)

In [0]:
display(df.limit(5))

In [0]:
df.printSchema()

In [0]:
df.write.format('delta') \
    .mode('overwrite') \
    .option('mergeSchema','true') \
    .saveAsTable(f'{catalog_name}.silver.slv_order_items')